# Data Quality Check and Cleansing
- Hourly Data
- Daily Data
- Cross Validation

In [1]:
import pandas as pd 
import numpy as np
import sys
from datetime import date
import holidays

## Hourly Data

In [2]:
# Hourly data
df_hour = pd.read_csv("../data/raw/Realisierter_Stromverbrauch_201901010000_202501010000_Stunde.csv", 
    delimiter = ';', 
    decimal = ',', 
    thousands = '.', 
    dtype={"Datum von": str, "Datum bis": str}
)

print("Hourly data - Columns")
for _ in list(df_hour.columns):
    print(f" - {_}")

df_hour.head(3)

Hourly data - Columns
 - Datum von
 - Datum bis
 - Netzlast [MWh] Berechnete Auflösungen
 - Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen
 - Pumpspeicher [MWh] Berechnete Auflösungen
 - Residuallast [MWh] Berechnete Auflösungen


,Datum von,Datum bis,Netzlast [MWh] Berechnete Auflösungen,Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Residuallast [MWh] Berechnete Auflösungen
0,01.01.2019 00:00,01.01.2019 01:00,43273.75,44546.75,1273.00,19825.50
1,01.01.2019 01:00,01.01.2019 02:00,41631.25,43042.00,1410.75,16465.75
2,01.01.2019 02:00,01.01.2019 03:00,40085.00,42561.75,2476.75,14456.25


### Column renaming and describing
- Descriptions are based on Page 41 ~ 44 of [SMARD user guide](smard-user-guide-09-2021-data.pdf).

In [3]:
columns = pd.DataFrame(
    [
        ("Datum von", "date_from", "datetime", "interval start date", ""),
        ("Datum bis", "date_to", "datetime", "interval end date", ""),
        ("Netzlast [MWh] Berechnete Auflösungen", "actual_consumption", "float", "consumption excluding hydro pumped storage", "MWh"),
        ("Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen", "actual_consumption_incl_pumped", "float", "actual_consumption + hydro_pumped_storage", "MWh"),
        ("Pumpspeicher [MWh] Berechnete Auflösungen", "hydro_pumped_storage", "float", "consumption of hydro pumped storage (subcategory of actual consumption)", "MWh"),
        ("Residuallast [MWh] Berechnete Auflösungen", "residual_load", "float", "actual_consumption minus the feed-in from photovoltaic, onshore wind and offshore wind installations", "MWh"),
    ],
    columns=["de", "en", "dtype", "desc", "unit"],
).set_index("en")


df_hour = df_hour.rename(
    columns = dict(
        zip(columns['de'], columns.index)
    )
)

df_hour.head()

,date_from,date_to,actual_consumption,actual_consumption_incl_pumped,hydro_pumped_storage,residual_load
0,01.01.2019 00:00,01.01.2019 01:00,43273.75,44546.75,1273.00,19825.50
1,01.01.2019 01:00,01.01.2019 02:00,41631.25,43042.00,1410.75,16465.75
2,01.01.2019 02:00,01.01.2019 03:00,40085.00,42561.75,2476.75,14456.25
3,01.01.2019 03:00,01.01.2019 04:00,39269.75,42554.25,3284.50,12101.50
4,01.01.2019 04:00,01.01.2019 05:00,39032.75,42748.50,3715.75,10134.25


### Semantic Validation

In [4]:
try:
    assert np.allclose(df_hour.actual_consumption + df_hour.hydro_pumped_storage, df_hour.actual_consumption_incl_pumped)
    print("Validated: actual consumption incl pumped = actual consumption + hydro pumped storage")
except:
    print("actual consumption incl pumped != actual consumption + hydro pumped storage")


Validated: actual consumption incl pumped = actual consumption + hydro pumped storage


### Data Cleansing
Map date column to `datetime` format 


In [5]:
# to datetime
df_hour["date_from"] = pd.to_datetime(df_hour["date_from"], dayfirst = True)
df_hour["date_to"] = pd.to_datetime(df_hour["date_to"], dayfirst = True)
df_hour.head(3)

,date_from,date_to,actual_consumption,actual_consumption_incl_pumped,hydro_pumped_storage,residual_load
0,2019-01-01 00:00:00,2019-01-01 01:00:00,43273.75,44546.75,1273.00,19825.50
1,2019-01-01 01:00:00,2019-01-01 02:00:00,41631.25,43042.00,1410.75,16465.75
2,2019-01-01 02:00:00,2019-01-01 03:00:00,40085.00,42561.75,2476.75,14456.25


### Data Duplicate Check
Duplication expected due to CEST.
Time spans from 2019 to 2025 so at most 6 duplicates are tolerated.

In [6]:
# Quality check 
# 1. Any hourly data row missing? 
adhoc = df_hour.groupby(df_hour['date_from'].dt.date).agg({'date_to': 'count'})

print('dates where hourly data is less than 24 rows (start of CEST)')
print(adhoc[adhoc['date_to'] < 24].index.astype(str).tolist())

print('\ndates where hourly data is more than 24 rows (end of CEST)')
print(adhoc[adhoc['date_to'] > 24].index.astype(str).tolist())

# 2. Any duplicated date? 
dup = df_hour.duplicated(subset = ['date_from']).sum()
print(f'''\nDuplicated date: {dup}''')

if dup == 6:
    print(f'These data rows come from the end of CEST')

dates where hourly data is less than 24 rows (start of CEST)
['2019-03-31', '2020-03-29', '2021-03-28', '2022-03-27', '2023-03-26', '2024-03-31']

dates where hourly data is more than 24 rows (end of CEST)
['2019-10-27', '2020-10-25', '2021-10-31', '2022-10-30', '2023-10-29', '2024-10-27']

Duplicated date: 6
These data rows come from the end of CEST


### Missing Value Check

In [7]:
df_hour.isna().sum()

date_from                         0
date_to                           0
actual_consumption                0
actual_consumption_incl_pumped    0
hydro_pumped_storage              0
residual_load                     0
dtype: int64

No missing values.

### Data Anomoly Check
- Check if there any negative value 
  - residual_load's negativity might be possible if reusable energy > actual consumption.

In [8]:
(df_hour[['actual_consumption', 'actual_consumption_incl_pumped', 'hydro_pumped_storage', 'residual_load']].apply(
    lambda x: x < 0
)
.sum()
)

actual_consumption                  0
actual_consumption_incl_pumped      0
hydro_pumped_storage                0
residual_load                     126
dtype: int64

### Feature Engineering (Preliminary)
- Generate additional features: 
  - day_of_week, 
  - month, 
  - is_holiday...
  
(This might be possibly useful for our model)

In [9]:
## * Enrichment: day of week
df_hour["day_of_week_raw"] = df_hour["date_from"].dt.dayofweek
df_hour["day_of_week"] = df_hour["date_from"].dt.day_name()
df_hour.head(3)[['date_from', 'day_of_week_raw', 'day_of_week']]

,date_from,day_of_week_raw,day_of_week
0,2019-01-01 00:00:00,1,Tuesday
1,2019-01-01 01:00:00,1,Tuesday
2,2019-01-01 02:00:00,1,Tuesday


In [10]:
## * Enrichment: month indicator
df_hour["month"] = df_hour["date_from"].dt.month
df_hour.head(3)[['date_from', 'month']]

,date_from,month
0,2019-01-01 00:00:00,1
1,2019-01-01 01:00:00,1
2,2019-01-01 02:00:00,1


In [11]:
## * Enrichment: holiday indicator (only national holidays are annotated.)
holidays_de = holidays.Germany(years=range(2019, 2026))
df_hour["is_holiday"] = df_hour["date_from"].apply(lambda x: x in holidays_de)
df_hour.head(3)[['date_from', 'is_holiday']]

,date_from,is_holiday
0,2019-01-01 00:00:00,True
1,2019-01-01 01:00:00,True
2,2019-01-01 02:00:00,True


## Daily Data
Daily Data is simply the aggregation of hourly data on date.
In addition to applying the previous data validation and cleansing steps, we further validate whether aggregating hourly data results in daily data.

In [12]:
# Daily data
df_day = pd.read_csv("../data/raw/Realisierter_Stromverbrauch_201901010000_202501010000_Tag.csv", 
    delimiter = ';', 
    decimal = ',', 
    thousands = '.', 
    dtype={"Datum von": str, "Datum bis": str}
)

print("Daily data - Columns")
for _ in list(df_day.columns):
    print(f" - {_}")

df_day.head(3)

Daily data - Columns
 - Datum von
 - Datum bis
 - Netzlast [MWh] Berechnete Auflösungen
 - Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen
 - Pumpspeicher [MWh] Berechnete Auflösungen
 - Residuallast [MWh] Berechnete Auflösungen


,Datum von,Datum bis,Netzlast [MWh] Berechnete Auflösungen,Netzlast inkl. Pumpspeicher [MWh] Berechnete Auflösungen,Pumpspeicher [MWh] Berechnete Auflösungen,Residuallast [MWh] Berechnete Auflösungen
0,01.01.2019,02.01.2019,1097118.75,1150739.75,53621.00,277981.25
1,02.01.2019,03.01.2019,1388494.25,1424715.50,36221.25,669048.50
2,03.01.2019,04.01.2019,1466308.25,1490488.00,24179.75,1150431.75


### Column renaming and describing

In [13]:
df_day = df_day.rename(
    columns = dict(
        zip(columns['de'], columns.index)
    )
)

df_day.head()

,date_from,date_to,actual_consumption,actual_consumption_incl_pumped,hydro_pumped_storage,residual_load
0,01.01.2019,02.01.2019,1097118.75,1150739.75,53621.00,277981.25
1,02.01.2019,03.01.2019,1388494.25,1424715.50,36221.25,669048.50
2,03.01.2019,04.01.2019,1466308.25,1490488.00,24179.75,1150431.75
3,04.01.2019,05.01.2019,1486876.25,1512694.00,25817.75,948617.25
4,05.01.2019,06.01.2019,1315755.00,1363757.75,48002.75,700158.25


### Semantically Validation

In [14]:
try:
    assert np.allclose(df_day.actual_consumption + df_day.hydro_pumped_storage, df_day.actual_consumption_incl_pumped)
    print("Validated: actual consumption incl pumped = actual consumption + hydro pumped storage")
except:
    print("actual consumption incl pumped != actual consumption + hydro pumped storage")


Validated: actual consumption incl pumped = actual consumption + hydro pumped storage


### Data Cleansing
- Map date column to `datetime` format 
- Data Quality check

In [15]:
df_day["date_from"] = pd.to_datetime(df_day["date_from"], dayfirst = True)
df_day["date_to"] = pd.to_datetime(df_day["date_to"], dayfirst = True)

### Data Duplicate Check

In [16]:
dup = df_day.duplicated(subset = ['date_from', 'date_to']).sum()
print(f'''\nDuplicated date: {dup}''')


Duplicated date: 0


### Missing Value Check

In [17]:
df_day.isna().sum()

date_from                         0
date_to                           0
actual_consumption                0
actual_consumption_incl_pumped    0
hydro_pumped_storage              0
residual_load                     0
dtype: int64

### Data Anomoly Check

In [18]:
(df_day[['actual_consumption', 'actual_consumption_incl_pumped', 'hydro_pumped_storage', 'residual_load']].apply(
    lambda x: x < 0
)
.sum()
)

actual_consumption                0
actual_consumption_incl_pumped    0
hydro_pumped_storage              0
residual_load                     0
dtype: int64

### Feature Engineering (Preliminary)
- Generate additional features: 
  - day_of_week, 
  - month, 
  - is_holiday...

In [19]:
## * Enrichment: day of week
df_day["day_of_week_raw"] = df_day["date_from"].dt.dayofweek
df_day["day_of_week"] = df_day["date_from"].dt.day_name()
df_day.head(3)[['date_from', 'day_of_week_raw', 'day_of_week']]

,date_from,day_of_week_raw,day_of_week
0,2019-01-01,1,Tuesday
1,2019-01-02,2,Wednesday
2,2019-01-03,3,Thursday


In [20]:
## * Enrichment: month indicator
df_day["month"] = df_day["date_from"].dt.month
df_day.head(3)[['date_from', 'month']]

,date_from,month
0,2019-01-01,1
1,2019-01-02,1
2,2019-01-03,1


In [21]:
## * Enrichment: holiday indicator (only national holidays are annotated.)
holidays_de = holidays.Germany(years=range(2019, 2026))
df_day["is_holiday"] = df_day["date_from"].apply(lambda x: x in holidays_de)
df_day.head(3)[['date_from', 'is_holiday']]

,date_from,is_holiday
0,2019-01-01,True
1,2019-01-02,False
2,2019-01-03,False


In [22]:
## * Enrichment: weekday indicator 
df_day["is_weekday"] = df_day.apply(
    lambda row: row['day_of_week_raw'] < 5, axis = 1
)

## Cross Validation
Check whether daily data = agg. sum of hourly data

In [23]:
hour_agg = (
    df_hour.drop(columns = "date_to")
    .groupby(df_hour['date_from'].dt.floor('D'))
    .agg({
        "actual_consumption": 'sum',
        "actual_consumption_incl_pumped": 'sum', 
        "hydro_pumped_storage": 'sum',
        "residual_load": 'sum',
        "day_of_week_raw": 'first',
        "day_of_week": 'first',
        "month": 'first',
        "is_holiday": 'first'
    })
)

cv = pd.merge(
    df_day, 
    hour_agg,
    left_on = df_day['date_from'].dt.floor('D'),
    right_index = True,
    suffixes = ['_daily', '_hourly'],
    validate = 'one_to_one'
)

# validation
num_cols = ['actual_consumption', 'actual_consumption_incl_pumped',
            'hydro_pumped_storage', 'residual_load']
cat_cols = ['day_of_week_raw', 'day_of_week', 'month', 'is_holiday']

for col in num_cols:
    d, h = cv[f'{col}_daily'], cv[f'{col}_hourly']
    ok = np.isclose(d, h, rtol=1e-6, atol=1e-3, equal_nan=True)
    print(f'{col}: {ok.sum()}/{len(ok)} match, max abs diff = {(d - h).abs().max():.6f}')
    if not ok.all():
        print(cv.loc[~ok, ['date_from', f'{col}_daily', f'{col}_hourly']].head())

for col in cat_cols:
    same = (cv[f'{col}_daily'] == cv[f'{col}_hourly'])
    print(f'{col}: {same.sum()}/{len(same)} match')


actual_consumption: 2192/2192 match, max abs diff = 0.020000
actual_consumption_incl_pumped: 2192/2192 match, max abs diff = 0.020000
hydro_pumped_storage: 2192/2192 match, max abs diff = 0.000000
residual_load: 2192/2192 match, max abs diff = 0.020000
day_of_week_raw: 2192/2192 match
day_of_week: 2192/2192 match
month: 2192/2192 match
is_holiday: 2192/2192 match
